# 19 · Faster eval with warm pools + tracing (any backend)

Cold starts dominate short eval tasks. A **warm pool** keeps sandboxes ready so
`acquire()` returns instantly; **OpenTelemetry** spans show the latency you save.

**Key concepts**
- **`WarmPool(size=N, **create_kwargs)`** — pre-starts sandboxes in background
  threads; `acquire()` hands one back immediately (or creates inline if empty)
  and replenishes. Client-side; works on any backend.
- **`observability.configure(tracer=...)`** — off by default and dependency-free;
  once enabled, `create` / `run_code` / `run_command` emit spans tagged with
  backend + sandbox id. Here we inject a tiny in-notebook tracer to read timings.
- Runs on **`agentcore-ci`** — no image needed.

**Prerequisites**
- AWS creds + region; Bedrock not required (tasks are fixed code snippets).

In [ ]:
%pip install -q raitbox

### Configuration + a minimal span collector

In [ ]:
import time
from contextlib import contextmanager

from raitbox import Sandbox, WarmPool
from raitbox.core import observability


class Span:
    def __init__(self, name):
        self.name, self.attrs, self.dur = name, {}, 0.0
    def set_attribute(self, k, v):
        self.attrs[k] = v
    def __enter__(self):
        self._t = time.perf_counter(); return self
    def __exit__(self, *e):
        self.dur = time.perf_counter() - self._t


class Tracer:
    def __init__(self):
        self.spans = []
    def start_as_current_span(self, name):
        s = Span(name); self.spans.append(s); return s


tracer = Tracer()
observability.configure(tracer=tracer)

TASKS = ["print(sum(range(100)))", "print(len('raitbox'))", "print(2 ** 10)", "print('ok')"]

## 1 · Baseline — a fresh sandbox per task
Each task pays the full create cost.

In [ ]:
t0 = time.perf_counter()
for code_str in TASKS:
    with Sandbox.create(backend="agentcore-ci") as sbx:
        sbx.run_code(code_str)
cold = time.perf_counter() - t0
print(f"cold path: {cold:.1f}s for {len(TASKS)} tasks")

## 2 · Warm pool — sandboxes created ahead of time
`acquire()` returns a ready sandbox; we kill each after use and the pool refills.

In [ ]:
t0 = time.perf_counter()
with WarmPool(size=len(TASKS), backend="agentcore-ci") as pool:
    time.sleep(0)  # (pool is already warming in the background)
    for code_str in TASKS:
        sbx = pool.acquire(timeout=120)
        try:
            sbx.run_code(code_str)
        finally:
            sbx.kill()
warm = time.perf_counter() - t0
print(f"warm path: {warm:.1f}s for {len(TASKS)} tasks")

## 3 · What the spans tell us

In [ ]:
from collections import defaultdict

agg = defaultdict(lambda: [0, 0.0])
for s in tracer.spans:
    agg[s.name][0] += 1
    agg[s.name][1] += s.dur
for name, (n, total) in sorted(agg.items()):
    print(f"{name:<28} count={n:<3} total={total:6.2f}s  avg={total/n:5.2f}s")

observability.reset()

## Recap
- The warm pool overlaps sandbox creation with useful work; `create` spans still
  happen but off the critical path.
- Enabling tracing needed one `configure` call and zero changes to the eval code.
- Point `configure()` at a real OTLP exporter to ship spans to your collector.

**Related:** `07-pass-at-k`, `10-parallel-eval-runner`, `docs/CONCEPTS.md` (Observability).